# 09 · Revision experiments (answers to the reviewer)

| Step | What | Where it runs | Time |
|---|---|---|---|
| R3 | Tip-Adapter baseline + **lab + field training** (new question: do lab images still help once you have field images?) | CPU | ~1 h |
| Features | All 30,978 PlantVillage training images for BioCLIP 2 and DINOv2-B | GPU | ~5–6 h |
| R2 | Lab classifier trained on **all** PlantVillage images vs the 200-per-class subset | CPU | ~1 h |
| R1 | Fine-tuned CNNs: epoch chosen on a **lab validation set**, **3 seeds** | GPU | ~6–7 h |

Suggested schedule: R3 now → features overnight → R2 next morning → R1 the next night.
Every cell can be stopped and re-run; it continues where it stopped. Charger in, sleep off for the GPU steps.
None of these steps changes the paper's original results — everything is written to `results/revision/`.

In [ ]:
import sys
from pathlib import Path
here = Path.cwd()
sys.path.insert(0, str(here if (here / "nbtools.py").exists() else here / "notebooks"))
from nbtools import run, show, figure, where, ROOT
where()

## R3 · Tip-Adapter and lab + field training (CPU, ~1 h)

In [ ]:
run("revision_baselines.py")

In [ ]:
show("results/revision/baselines/tip_vs_v2.csv")
show("results/revision/baselines/tip_by_dataset.csv")

## Full PlantVillage features (GPU, ~5–6 h) — one cell per model

In [ ]:
run("extract_features.py", "--models", "bioclip2", "--splits", "plantvillage_train", "plantvillage_test", "--workers", 2)

In [ ]:
run("extract_features.py", "--models", "dinov2_b14", "--splits", "plantvillage_train", "plantvillage_test", "--workers", 2)

## R2 · All lab images vs subset (CPU, ~1 h)

In [ ]:
run("revision_fulllab.py")

In [ ]:
show("results/revision/fulllab/full_vs_subset.csv")

## R1 · Fine-tuned CNNs with validation set and 3 seeds (GPU, ~6–7 h)
One cell per CNN so you can run them on different evenings. Field accuracy is printed after every epoch for the
analysis only — the epoch is chosen on the lab **validation** set.

In [ ]:
run("revision_finetune.py", "--models", "resnet50")

In [ ]:
run("revision_finetune.py", "--models", "effnet_b0")

In [ ]:
run("revision_finetune.py", "--models", "mobilenet_v3")

In [ ]:
show("results/revision/finetune/summary.csv")

## Send the results
Run the cell below, then send the file `revision_results.zip` (in the project folder) to Claude.

In [ ]:
import shutil
shutil.make_archive(str(ROOT / "revision_results"), "zip", ROOT / "results", "revision")
print("Created", ROOT / "revision_results.zip")